In [1]:
import numpy as np
from pathlib import Path
import pickle
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)
from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar


# =============================================================================
# Paths
# =============================================================================

VIT_LOGITS_PATH = (
    "/home/maria/ProjectionSort/data/"
    "google_vit-base-patch16-224_embeddings_logits.pkl"
)

NEURAL_PATH = "/home/maria/Science/data/hybrid_neural_responses_reduced.npy"
HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"

OUTDIR = Path("/home/maria/Science/results/human_vs_vit_adam_angles")
OUTDIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Config: matches your original script
# =============================================================================

LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 3000


# =============================================================================
# Loading labels and neural data
# =============================================================================

def load_vit_array(vit_logits_path: str) -> np.ndarray:
    path = Path(vit_logits_path)

    if not path.exists():
        raise FileNotFoundError(f"ViT logits file not found: {path}")

    if path.suffix == ".npz":
        obj = np.load(path, allow_pickle=True)
        if "natural_scenes" not in obj:
            raise KeyError(f"Expected key 'natural_scenes'. Found keys: {list(obj.keys())}")
        vit = obj["natural_scenes"]

    elif path.suffix == ".npy":
        vit = np.load(path, allow_pickle=True)

    elif path.suffix in [".pkl", ".pickle"]:
        with open(path, "rb") as f:
            obj = pickle.load(f)

        if isinstance(obj, dict):
            if "natural_scenes" not in obj:
                raise KeyError(f"Expected key 'natural_scenes'. Found keys: {list(obj.keys())}")
            vit = obj["natural_scenes"]
        else:
            vit = obj

    else:
        raise ValueError(f"Unsupported ViT file extension: {path.suffix}")

    vit = np.asarray(vit)

    if vit.ndim != 2 or vit.shape[1] != 1000:
        raise ValueError(f"Expected ViT logits shape (n_images, 1000), got {vit.shape}")

    return vit


def load_vit_animate_labels(vit_logits_path: str) -> np.ndarray:
    vit = load_vit_array(vit_logits_path)
    top1 = np.argmax(vit, axis=1)

    # ImageNet convention:
    # classes 0..397 are treated as animate.
    image_labels = (top1 <= 397).astype(np.int64)

    print(f"Loaded ViT logits: {vit.shape}")
    print(f"ViT label counts [inanimate, animate]: {np.bincount(image_labels, minlength=2)}")
    print("Convention: 0 = inanimate, 1 = animate")

    return image_labels


def load_human_labels(path: str) -> np.ndarray:
    labels = np.load(path, allow_pickle=True).item()["labels"]
    labels = np.asarray(labels).astype(np.int64)

    print(f"Loaded human labels: {labels.shape}")
    print("Human label counts [inanimate, animate], excluding -1:")
    print(np.bincount(labels[labels != -1], minlength=2))

    return labels


def load_labels(label_source: str) -> np.ndarray:
    if label_source == "human":
        return load_human_labels(HUMAN_LABEL_PATH)

    if label_source == "vit":
        return load_vit_animate_labels(VIT_LOGITS_PATH)

    raise ValueError("label_source must be 'human' or 'vit'")


def load_clean_neural_and_labels(label_source: str):
    X = np.load(NEURAL_PATH, allow_pickle=True)
    X = np.asarray(X)

    y = load_labels(label_source)

    print()
    print("=" * 80)
    print(f"Loading data for label source: {label_source}")
    print("=" * 80)
    print(f"Raw neural shape: {X.shape}")
    print(f"Labels shape:     {y.shape}")

    if X.shape[0] != len(y) and X.shape[1] == len(y):
        print("[INFO] Transposing neural matrix to images x features.")
        X = X.T

    if X.shape[0] != len(y):
        raise ValueError(f"Expected rows to match labels. Got X={X.shape}, y={y.shape}")

    mask = y != -1
    X = X[mask]
    y = y[mask].astype(np.int64)

    finite_cols = np.all(np.isfinite(X), axis=0)
    nonconstant_cols = np.std(X[:, finite_cols], axis=0) > 1e-12

    good_cols = np.zeros(X.shape[1], dtype=bool)
    good_cols[np.where(finite_cols)[0][nonconstant_cols]] = True

    X = X[:, good_cols]

    print(f"Clean neural shape: {X.shape}")
    print(f"Removed bad/nonconstant columns: {np.sum(~good_cols)}")
    print(f"Final label counts [inanimate, animate]: {np.bincount(y, minlength=2)}")

    return X, y


# =============================================================================
# Math helpers
# =============================================================================

def sigmoid_np(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def unit(v, eps=1e-12):
    v = np.asarray(v, dtype=np.float64)
    n = np.linalg.norm(v)
    if n < eps:
        raise ValueError("Vector has near-zero norm.")
    return v / n


def angle_degrees(u, v):
    u = unit(u)
    v = unit(v)

    cos = float(np.dot(u, v))
    cos = float(np.clip(cos, -1.0, 1.0))

    angle = float(np.degrees(np.arccos(cos)))
    return cos, angle


# =============================================================================
# Adam logistic regression: original setup
# =============================================================================

class TorchLogisticRegression(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)


def fit_adam_logistic_axis(
    X_train,
    y_train,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    seed=0,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_t = torch.tensor(X_train.astype(np.float32))
    y_t = torch.tensor(y_train.astype(np.float32)).view(-1, 1)

    model = TorchLogisticRegression(X_train.shape[1])

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    loss_fn = torch.nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X_t)
        loss = loss_fn(logits, y_t)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        w = model.linear.weight.detach().cpu().numpy().ravel()
        b = float(model.linear.bias.detach().cpu().numpy()[0])

    return w, b


def loo_adam_scores(X, y, label_source: str):
    n, d = X.shape

    scores = np.zeros(n, dtype=np.float64)
    probs = np.zeros(n, dtype=np.float64)
    preds = np.zeros(n, dtype=np.int64)
    biases = np.zeros(n, dtype=np.float64)
    dirs = np.zeros((n, d), dtype=np.float32)

    for test_idx in range(n):
        train_idx = np.arange(n) != test_idx

        X_train_raw = X[train_idx]
        y_train = y[train_idx]

        X_test_raw = X[~train_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_test = scaler.transform(X_test_raw)

        # Important: matches your original script.
        w, b = fit_adam_logistic_axis(
            X_train,
            y_train,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            epochs=EPOCHS,
            seed=test_idx,
        )

        logit = float(X_test[0] @ w + b)
        prob = float(sigmoid_np(logit))
        pred = int(prob >= 0.5)

        scores[test_idx] = logit
        probs[test_idx] = prob
        preds[test_idx] = pred
        biases[test_idx] = b
        dirs[test_idx] = unit(w).astype(np.float32)

        print(
            f"[{label_source} Adam LOO {test_idx + 1:03d}/{n}] "
            f"true={y[test_idx]} logit={logit:+.6f} prob={prob:.4f} pred={pred}"
        )

    return scores, probs, preds, biases, dirs


def summarize_adam(y, probs, preds, label_source: str):
    acc = accuracy_score(y, preds)
    bal_acc = balanced_accuracy_score(y, preds)
    auc = roc_auc_score(y, probs)
    cm = confusion_matrix(y, preds, labels=[0, 1])

    print()
    print("=" * 80)
    print(f"LOO Adam logistic axis: {label_source} labels")
    print("=" * 80)
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"AUC:               {auc:.4f}")
    print("Confusion matrix rows=true [inanimate, animate], cols=pred:")
    print(cm)

    return {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "auc": auc,
        "confusion_matrix": cm,
    }


def run_one_label_source(label_source: str):
    X, y = load_clean_neural_and_labels(label_source)

    print()
    print("#" * 80)
    print(f"Running LOO Adam logistic regression for {label_source} labels")
    print("#" * 80)

    adam_scores, adam_probs, adam_preds, adam_biases, adam_dirs = loo_adam_scores(
        X,
        y,
        label_source=label_source,
    )

    metrics = summarize_adam(
        y,
        adam_probs,
        adam_preds,
        label_source=label_source,
    )

    outpath = OUTDIR / f"adam_{label_source}_labels_results.npz"

    np.savez_compressed(
        outpath,
        y=y,
        adam_scores=adam_scores,
        adam_probs=adam_probs,
        adam_preds=adam_preds,
        adam_biases=adam_biases,
        adam_dirs=adam_dirs,
        accuracy=metrics["accuracy"],
        balanced_accuracy=metrics["balanced_accuracy"],
        auc=metrics["auc"],
        confusion_matrix=metrics["confusion_matrix"],
    )

    print("Saved:", outpath)

    return {
        "label_source": label_source,
        "y": y,
        "adam_scores": adam_scores,
        "adam_probs": adam_probs,
        "adam_preds": adam_preds,
        "adam_biases": adam_biases,
        "adam_dirs": adam_dirs,
        "metrics": metrics,
        "outpath": outpath,
    }


# =============================================================================
# Compare accuracies
# =============================================================================

def compare_human_vs_vit_adam_accuracy(human_res, vit_res):
    human_y = human_res["y"]
    vit_y = vit_res["y"]

    human_preds = human_res["adam_preds"]
    vit_preds = vit_res["adam_preds"]

    human_correct = human_preds == human_y
    vit_correct = vit_preds == vit_y

    d = human_correct.astype(int) - vit_correct.astype(int)

    print()
    print("#" * 80)
    print("Human-label Adam vs ViT-label Adam accuracy")
    print("#" * 80)

    print(f"Human Adam accuracy: {human_correct.mean():.4f}")
    print(f"ViT Adam accuracy:   {vit_correct.mean():.4f}")
    print(f"Difference:          {d.mean():+.4f}")
    print(f"Extra correct images human - ViT: {d.sum()}")

    human_only = np.sum((human_correct == 1) & (vit_correct == 0))
    vit_only = np.sum((human_correct == 0) & (vit_correct == 1))
    both_correct = np.sum((human_correct == 1) & (vit_correct == 1))
    both_wrong = np.sum((human_correct == 0) & (vit_correct == 0))

    print()
    print(f"Human-only correct: {human_only}")
    print(f"ViT-only correct:   {vit_only}")
    print(f"Both correct:       {both_correct}")
    print(f"Both wrong:         {both_wrong}")

    t_stat, p_two = ttest_1samp(d, popmean=0)
    p_one_human_greater = p_two / 2 if t_stat > 0 else 1 - p_two / 2

    print()
    print("Paired t-test on image-level correctness difference")
    print(f"t-stat: {t_stat:.6f}")
    print(f"two-sided p: {p_two:.6f}")
    print(f"one-sided p, human Adam > ViT Adam: {p_one_human_greater:.6f}")

    table = np.array([
        [
            np.sum((human_correct == 0) & (vit_correct == 0)),
            np.sum((human_correct == 0) & (vit_correct == 1)),
        ],
        [
            np.sum((human_correct == 1) & (vit_correct == 0)),
            np.sum((human_correct == 1) & (vit_correct == 1)),
        ],
    ])

    print()
    print("McNemar table")
    print("rows = human Adam wrong/correct")
    print("cols = ViT Adam wrong/correct")
    print(table)

    mc = mcnemar(table, exact=True)
    print(f"McNemar exact p-value: {mc.pvalue:.6f}")

    rng = np.random.default_rng(0)
    B = 10_000
    boot = np.zeros(B)

    n = len(d)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        boot[b] = d[idx].mean()

    lo, hi = np.percentile(boot, [2.5, 97.5])

    print()
    print("Bootstrap")
    print(f"Observed accuracy difference: {d.mean():+.4f}")
    print(f"95% bootstrap CI: ({lo:+.4f}, {hi:+.4f})")
    print(f"Bootstrap p(diff <= 0): {np.mean(boot <= 0):.6f}")

    outpath = OUTDIR / "human_vs_vit_adam_accuracy_comparison.npz"

    np.savez_compressed(
        outpath,
        human_correct=human_correct,
        vit_correct=vit_correct,
        correctness_difference=d,
        mcnemar_table=table,
        mcnemar_p=mc.pvalue,
        ttest_t=t_stat,
        ttest_p_two_sided=p_two,
        ttest_p_one_sided_human_greater=p_one_human_greater,
        bootstrap_ci_95=np.array([lo, hi]),
        bootstrap_p_diff_leq_0=np.mean(boot <= 0),
    )

    print("Saved accuracy comparison:", outpath)


# =============================================================================
# Compare decoder angles
# =============================================================================

def compare_human_vs_vit_decoder_angles(human_res, vit_res):
    human_dirs = human_res["adam_dirs"].astype(np.float64)
    vit_dirs = vit_res["adam_dirs"].astype(np.float64)

    if human_dirs.shape != vit_dirs.shape:
        raise ValueError(f"Shape mismatch: human {human_dirs.shape}, vit {vit_dirs.shape}")

    n = human_dirs.shape[0]

    fold_cosines = np.zeros(n, dtype=np.float64)
    fold_angles = np.zeros(n, dtype=np.float64)

    for i in range(n):
        cos, angle = angle_degrees(human_dirs[i], vit_dirs[i])
        fold_cosines[i] = cos
        fold_angles[i] = angle

    mean_human_dir = unit(human_dirs.mean(axis=0))
    mean_vit_dir = unit(vit_dirs.mean(axis=0))

    mean_cos, mean_angle = angle_degrees(mean_human_dir, mean_vit_dir)

    print()
    print("#" * 80)
    print("Angle between human-label Adam decoder and ViT-label Adam decoder")
    print("#" * 80)

    print()
    print("Foldwise angle, comparing same LOO fold")
    print(f"Mean cosine:    {fold_cosines.mean():.4f}")
    print(f"Median cosine:  {np.median(fold_cosines):.4f}")
    print(f"Min cosine:     {fold_cosines.min():.4f}")
    print(f"Max cosine:     {fold_cosines.max():.4f}")

    print()
    print(f"Mean angle:     {fold_angles.mean():.2f} degrees")
    print(f"Median angle:   {np.median(fold_angles):.2f} degrees")
    print(f"Min angle:      {fold_angles.min():.2f} degrees")
    print(f"Max angle:      {fold_angles.max():.2f} degrees")

    print()
    print("Angle between mean decoder directions")
    print(f"Cosine:         {mean_cos:.4f}")
    print(f"Angle:          {mean_angle:.2f} degrees")

    outpath = OUTDIR / "human_vs_vit_adam_decoder_angles.npz"

    np.savez_compressed(
        outpath,
        fold_cosines=fold_cosines,
        fold_angles=fold_angles,
        mean_human_dir=mean_human_dir,
        mean_vit_dir=mean_vit_dir,
        mean_cosine=mean_cos,
        mean_angle=mean_angle,
    )

    print("Saved decoder angle comparison:", outpath)


# =============================================================================
# Main
# =============================================================================

def main():
    human_res = run_one_label_source("human")
    vit_res = run_one_label_source("vit")

    compare_human_vs_vit_adam_accuracy(human_res, vit_res)
    compare_human_vs_vit_decoder_angles(human_res, vit_res)


if __name__ == "__main__":
    main()

Loaded human labels: (118,)
Human label counts [inanimate, animate], excluding -1:
[62 56]

Loading data for label source: human
Raw neural shape: (39209, 118)
Labels shape:     (118,)
[INFO] Transposing neural matrix to images x features.
Clean neural shape: (118, 39209)
Removed bad/nonconstant columns: 0
Final label counts [inanimate, animate]: [62 56]

################################################################################
Running LOO Adam logistic regression for human labels
################################################################################
[human Adam LOO 001/118] true=1 logit=-2.554781 prob=0.0721 pred=0
[human Adam LOO 002/118] true=1 logit=-1.164098 prob=0.2379 pred=0
[human Adam LOO 003/118] true=1 logit=+12.931960 prob=1.0000 pred=1
[human Adam LOO 004/118] true=1 logit=+2.839264 prob=0.9448 pred=1
[human Adam LOO 005/118] true=1 logit=-5.751934 prob=0.0032 pred=0
[human Adam LOO 006/118] true=1 logit=+16.565349 prob=1.0000 pred=1
[human Adam LOO 007/11